# CSE 151B Competition — Optimized Notebook

Improvements over the starter notebook (first 10 questions for quick evaluation):

| # | Change | Impact |
|---|---|---|
| 1 | Runs 10 questions (not hardcoded 5) | Bug fix |
| 2 | `MAX_TOKENS=12000` to fit within `max_model_len=16384` | Prevents truncation |
| 3 | Better MCQ letter extraction (answer-statement patterns first) | Accuracy |
| 4 | MCQ system prompt tuned for thinking model | Accuracy |
| 5 | Majority voting with N=5 samples (MCQ) | High ROI |
| 6 | `enable_prefix_caching=True` | Free speedup |
| 7 | `gpu_memory_utilization=0.80` | Longer context / throughput |
| 8 | `temperature=0.7, top_p=0.8` per Qwen3-Thinking docs | Better diversity |
| 9 | Few-shot math example in free-form system prompt | Format anchoring |
| 10 | Enforce single `\\boxed{}` in free-form prompt | Judger compatibility |

## 1. Environment Setup

Same as starter. Comment out the install block after first run, then restart the kernel.

In [1]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create venv and install — use full path since PATH isn't updated yet
# !~/.local/bin/uv venv .venv --seed
# !~/.local/bin/uv pip install --python .venv/bin/python sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")

In [2]:
# Activate venv — run every time
!source ./.venv/bin/activate

## 2. Imports & Configuration

Key changes vs starter:
- `MAX_TOKENS = 12000` — fits within `max_model_len=16384` (leaves ~4k for prompt)
- `N_SAMPLES = 5` — enables majority voting
- `N_QUESTIONS = 10` — run first 10 questions for quick evaluation

In [3]:
import json
import os
import re
import sys
from collections import Counter
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID     = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID       = "0"                   # CUDA_VISIBLE_DEVICES
DATA_PATH    = "data/public.jsonl"
OUTPUT_PATH  = "results/optimized_results.jsonl"
MAX_TOKENS   = 12000   # Fix #2: lowered to fit within max_model_len=16384
N_SAMPLES    = 5       # Fix #5: majority voting sample count
N_QUESTIONS  = 10      # First N questions to evaluate

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 3. Load the Dataset

In [4]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")
print(f"Will evaluate first {N_QUESTIONS} questions.")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)
Will evaluate first 10 questions.

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

Changes vs starter:
- **Fix #4**: MCQ prompt now tells the model to think step-by-step (works *with* the thinking model instead of against it)
- **Fix #9 & #10**: Free-form prompt includes a worked example and explicitly says to use exactly one `\\boxed{}`

In [5]:
# Fix #4: MCQ prompt tuned for thinking model — lean into reasoning, don't suppress it
SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Think step by step to evaluate each option carefully. "
    "At the end, state your final answer as a single letter inside \\boxed{}, e.g. \\boxed{C}."
)

# Fix #9 & #10: free-form prompt with few-shot example and single-\boxed enforcement
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Use exactly one \\boxed{} containing all final answers separated by commas, "
    "e.g. \\boxed{3, 7} for two sub-answers or \\boxed{42} for a single answer. "
    "Do NOT use multiple \\boxed{} blocks.\n\n"
    "Example:\n"
    "Problem: Find the roots of x^2 - 5x + 6 = 0.\n"
    "Solution: Factor as (x-2)(x-3)=0, so x=2 or x=3.\n"
    "\\boxed{2, 3}"
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} system prompt ──")
    print(sys_p)
    print(f"\n── {label} user prompt (first 300 chars) ──")
    print(usr_p[:300], "...\n")

── MCQ system prompt ──
You are an expert mathematician. Think step by step to evaluate each option carefully. At the end, state your final answer as a single letter inside \boxed{}, e.g. \boxed{C}.

── MCQ user prompt (first 300 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. $frac{1}{a^2}$ ...

── Free-form system prompt ──
You are an expert mathematician. Solve the problem step-by-step. Use exactly one \boxed{} containing all final answers separated by commas, e.g. \boxed{3, 7} for two sub-answers or \boxed{42} for a single answer. Do NOT use multiple \boxed{} blocks.

Example:
Problem: Find the roots of x^2 - 5x + 6 = 0.
Solution: Factor as (x-2)(x-3)=0, so x=2 or x=3.
\boxed{2, 3}

── Free-form user prompt (first 300 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM

Changes vs starter:
- **Fix #6**: `enable_prefix_caching=True` — caches shared system-prompt KV for free speedup
- **Fix #7**: `gpu_memory_utilization=0.65` — up from 0.50; `enforce_eager=True` avoids CUDA graph warmup OOM
- **Fix #8**: `temperature=0.7, top_p=0.8` — per Qwen3-Thinking documentation
- **Fix #5**: `n=N_SAMPLES` — generates 5 completions per prompt for majority voting

> **Note on `enforce_eager=True`**: vLLM's CUDA graph capture profiles graphs at up to 512 batch sizes, causing a brief memory spike that can OOM even at moderate `gpu_memory_utilization`. Eager mode skips this entirely; throughput impact is negligible for small question batches.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=True,   # Fix #6: cache shared system-prompt KV
    gpu_memory_utilization=0.5,  # Fix #7: up from 0.50; safe with enforce_eager
    max_model_len=16384,
    trust_remote_code=True,
    enforce_eager=True,           # Prevents CUDA graph warmup OOM on memory-limited GPUs
    max_num_seqs=64,
)

sampling_params = SamplingParams(
    n=N_SAMPLES,          # Fix #5: generate N_SAMPLES completions per prompt
    max_tokens=MAX_TOKENS,
    temperature=0.7,      # Fix #8: Qwen3-Thinking recommended value
    top_p=0.8,            # Fix #8: Qwen3-Thinking recommended value
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

INFO 05-10 02:56:58 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 16384, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.5, 'max_num_seqs': 64, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'enforce_eager': True, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 05-10 02:56:59 [model.py:555] Resolved architecture: Qwen3ForCausalLM
INFO 05-10 02:56:59 [model.py:1680] Using max model len 16384
INFO 05-10 02:56:59 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-10 02:57:00 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-10 02:57:00 [nixl_utils.py:34] NIXL is not available
WARNING 05-10 02:57:00 [nixl_utils.py:44] NIXL agent config is not available
INFO 05-10 02:57:00 [vllm.py:840] Asynchronous scheduling is enabled.
WARNING 05-10 02:57:00 [vllm.py:896] Enforce eager set, disabling torch.compi

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=14280) /home/cow003/private/151B_SP26_Competition/.venv/lib/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
(EngineCore pid=14280)   torch._check_is_size(blocksize)


(EngineCore pid=14280) INFO 05-10 02:57:09 [gpu_model_runner.py:4879] Model loading took 2.7 GiB memory and 4.570066 seconds
(EngineCore pid=14280) INFO 05-10 02:57:12 [gpu_worker.py:440] Available KV cache memory: 8.3 GiB
(EngineCore pid=14280) INFO 05-10 02:57:12 [kv_cache_utils.py:1708] GPU KV cache size: 60,448 tokens
(EngineCore pid=14280) INFO 05-10 02:57:12 [kv_cache_utils.py:1709] Maximum concurrency for 16,384 tokens per request: 3.69x


(EngineCore pid=14280) 2026-05-10 02:57:12,286 - INFO - autotuner.py:457 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=14280) 2026-05-10 02:57:12,370 - INFO - autotuner.py:466 - flashinfer.jit: [Autotuner]: Autotuning process ends


(EngineCore pid=14280) INFO 05-10 02:57:13 [core.py:306] init engine (profile, create kv cache, warmup model) took 4.24 s
(EngineCore pid=14280) WARNING 05-10 02:57:14 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=14280) WARNING 05-10 02:57:14 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=14280) INFO 05-10 02:57:14 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'])
(EngineCore pid=14280) INFO 05-10 02:57:14 [vllm.py:1089] Cudagraph is disabled under eager mode
Model loaded.


(EngineCore pid=14280) /home/cow003/private/151B_SP26_Competition/.venv/lib/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
(EngineCore pid=14280)   torch._check_is_size(blocksize)


## 6. Generate Responses

Changes vs starter:
- Runs `data[:N_QUESTIONS]` (first 10) instead of hardcoded `data[:5]`
- Each prompt produces `N_SAMPLES=5` completions for majority voting

In [7]:
# Build prompts for first N_QUESTIONS entries
subset = data[:N_QUESTIONS]
prompts = []
for item in subset:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate — each output has N_SAMPLES completions
print(f"Generating {N_SAMPLES} samples x {len(prompts)} questions = {N_SAMPLES * len(prompts)} total completions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

# all_responses[i] is a list of N_SAMPLES response strings for question i
all_responses = [[o.text.strip() for o in out.outputs] for out in outputs]

# Preview first 3 questions (first sample only)
for i in range(min(3, len(all_responses))):
    print(f"\n── Response {i} (id={subset[i].get('id')}, sample 0) ──")
    print(all_responses[i][0][:400], "..." if len(all_responses[i][0]) > 400 else "")

Generating 5 samples x 10 questions = 50 total completions...


Rendering prompts:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


── Response 0 (id=0, sample 0) ──
Okay, let's see. I need to find the sum of the first 325 positive even whole numbers. Hmm, first, let me make sure I understand what the first 325 positive even whole numbers are. Positive even whole numbers start from 2, right? So the first one is 2, the second is 4, the third is 6, and so on. So the nth positive even whole number is 2n. Let me confirm that. For n=1, it's 2*1=2, n=2, 2*2=4, yeah, ...

── Response 1 (id=1, sample 0) ──
Okay, let's try to figure out this integral. The problem is the integral from negative infinity to positive infinity of (a^(3/2)) divided by (s² + a²) ds. Hmm, first, I need to recall how to integrate functions like 1/(s² + a²). 

Wait, the integral of 1/(s² + a²) ds is (1/a) arctan(s/a) + C, right? Because the derivative of arctan(x) is 1/(1 + x²), so scaling it up, the integral of 1/(s² + a²) is ...

── Response 2 (id=2, sample 0) ──
Okay, let's try to solve this problem step by step. First, part (a) is about the temp

## 7. Score Responses

Changes vs starter:
- **Fix #3**: `extract_letter()` now searches for explicit answer-statement patterns before falling back to last capital letter — avoids confusing letters mentioned during reasoning
- **Fix #5**: MCQ scoring uses majority vote across all `N_SAMPLES` completions
- Free-form scoring uses the first completion (majority voting on symbolic answers is non-trivial)

In [8]:
# Fix #3: look for explicit answer-statement patterns before last-capital fallback
_ANSWER_PATTERNS = [
    r"\\boxed\{([A-Za-z])\}",                        # \boxed{X}
    r"(?:the\s+)?answer\s+is\s+[\(\[]?([A-Z])[\)\]]?",  # answer is X / (X)
    r"(?:correct\s+)?answer\s*[:\-]\s*[\(\[]?([A-Z])[\)\]]",  # answer: X
    r"(?:therefore|thus|so),?\s+(?:the\s+)?(?:answer|choice|option)\s+is\s+[\(\[]?([A-Z])[\)\]]",
    r"option\s+([A-Z])\s+is\s+correct",
    r"select\s+(?:option\s+)?([A-Z])",
    r"choose\s+(?:option\s+)?([A-Z])",
]

def extract_letter(text: str) -> str:
    # Try each high-confidence pattern first
    for pat in _ANSWER_PATTERNS:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.group(1).upper()
    # Last-resort: last isolated capital letter in the text
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq_majority(response_set: list[str], gold_letter: str) -> tuple[bool, str]:
    """Majority-vote across N_SAMPLES and return (correct, predicted_letter)."""
    votes = [extract_letter(r) for r in response_set]
    votes = [v for v in votes if v]  # drop empty extractions
    if not votes:
        return False, ""
    pred = Counter(votes).most_common(1)[0][0]
    return pred == gold_letter.strip().upper(), pred


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response_set in tqdm(zip(subset, all_responses), total=len(subset), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct, pred_letter = score_mcq_majority(response_set, str(gold))
        response = response_set[0]   # store sample 0 for inspection
        meta = {"votes": [extract_letter(r) for r in response_set], "pred": pred_letter}
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        response  = response_set[0]  # use first sample for free-form
        meta      = {}
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
        **meta,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|██████████| 10/10 [00:00<00:00, 18.38it/s]

Scoring complete. 10 results.


## 8. Summary

In [9]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS (first 10 questions)")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

# Show majority-vote details for MCQ
if mcq_res:
    print("\n── MCQ majority-vote breakdown ──")
    for r in mcq_res:
        votes = r.get("votes", [])
        pred  = r.get("pred", "?")
        mark  = "✓" if r["correct"] else "✗"
        print(f"  id={r['id']:4d}  gold={r['gold']}  votes={votes}  pred={pred}  {mark}")

EVALUATION RESULTS (first 10 questions)
  MCQ        :    2 /    3  (66.67%)
  Free-form  :    3 /    7  (42.86%)
  Overall    :    5 /   10  (50.00%)

── MCQ majority-vote breakdown ──
  id=   1  gold=F  votes=['B', 'E', 'B', 'E', 'B']  pred=B  ✗
  id=   4  gold=C  votes=['C', 'C', 'C', 'C', 'C']  pred=C  ✓
  id=   9  gold=A  votes=['A', 'A', 'A', 'A', 'A']  pred=A  ✓


## 9. Save Results

In [10]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 10 records to results/optimized_results.jsonl


## Next Steps

To run the full dataset, change `N_QUESTIONS = 10` to `N_QUESTIONS = len(data)` in the Configuration cell.

Further improvements to try:
- Increase `N_SAMPLES` to 7–9 for stronger majority voting on MCQ
- Apply majority voting to free-form by extracting `\\boxed{}` across samples and picking the modal answer
- Fine-tune the model on math problems from similar distributions